# CI/CD 第2周：构建与测试 — Matrix、缓存、Artifacts 与 Secrets

> **学习目标**：掌握 Matrix Build、依赖缓存、Artifacts 管理和 Secrets 配置

---

## 开篇回顾

第1周你学会了写基本的 Workflow：push 触发、checkout、setup-python、多 Job 依赖。

但实际项目中的 CI 比这复杂得多：
- 需要在**多个 Python 版本**上测试兼容性
- 每次 CI 都要 pip install 太慢了——能不能**缓存依赖**？
- 测试报告、构建产物怎么**传递给下一个 Job**？
- API 密钥、Token 怎么安全地传入 Workflow？

这周我们逐一解决这些问题。

---

## Day 8：Matrix Strategy（矩阵策略）

### 为什么需要 Matrix？

一个 Python 库需要兼容 Python 3.10/3.11/3.12，有时还需要兼容 macOS 和 Windows。
如果每个组合写一个 Job，代码会膨胀到无法维护。

**Matrix Strategy 让你用一行配置生成多个 Job 组合**：

```yaml
strategy:
  matrix:
    python-version: ["3.10", "3.11", "3.12"]
    os: [ubuntu-latest, macos-latest]
```

上面的配置会产生 **3 × 2 = 6 个 Job**，每个 Job 自动获得对应的 `python-version` 和 `os` 值。

### 完整 Workflow 示例

```yaml
name: Matrix Test
on: push
jobs:
  test:
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
        os: [ubuntu-latest]
    runs-on: \${{ matrix.os }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: \${{ matrix.python-version }}
      - run: pip install -r requirements.txt
      - run: pytest
```

In [ ]:
# 在本地模拟 Matrix Strategy

print("=" * 60)
print("Matrix Strategy 模拟")
print("配置: python-version × os")
print("=" * 60)

matrix = {
    "python-version": ["3.10", "3.11", "3.12"],
    "os": ["ubuntu-latest", "macos-latest"]
}

# 笛卡尔积
combinations = []
for py in matrix["python-version"]:
    for os_name in matrix["os"]:
        combinations.append((py, os_name))

print(f"\n生成 {len(combinations)} 个 Job（{len(matrix['python-version'])} Python × {len(matrix['os'])} OS）")
print("-" * 40)

for i, (py, os_name) in enumerate(combinations, 1):
    print(f"Job #{i}: Python {py} | {os_name}")

print("\n所有 Job 并行执行！GitHub 会自动为每个组合创建一个独立的 Runner")

# 查看本地 Python 版本（模拟其中一个组合）
import sys
print(f"\n本地 Python 版本: {sys.version.split()[0]}")
print(f"这对应 matrix 中的: python-version: '{'.'.join(sys.version.split()[0].split('.')[:2])}'")

### fail-fast 与 include/exclude

```yaml
strategy:
  fail-fast: true  # 默认 true：一个 Job 失败就取消所有运行中的 Job
  matrix:
    python-version: ["3.10", "3.11", "3.12"]
    os: [ubuntu-latest]
    include:
      # 额外加一个 Windows + Python 3.12 的测试
      - os: windows-latest
        python-version: "3.12"
    exclude:
      # 排除某个不需要的组合
      - os: ubuntu-latest
        python-version: "3.10"
```

| 参数 | 说明 |
|------|------|
| `fail-fast` | true = 一个失败全部取消；false = 各自独立运行 |
| `include` | 添加默认组合之外的额外组合 |
| `exclude` | 排除不需要的组合（避免组合爆炸） |

**组合爆炸警告**：3 Python × 3 OS × 2 database = 18 个 Job！
在生产中要合理控制组合数量，避免浪费 Action 时长。

In [ ]:
# demonstrate include/exclude logic

print("组合生成逻辑演示")
print("=" * 40)

base_python = ["3.10", "3.11", "3.12"]
base_os = ["ubuntu-latest"]

include = [{"os": "windows-latest", "python-version": "3.12"}]
exclude = [{"os": "ubuntu-latest", "python-version": "3.10"}]

# 生成组合
matrix = []
for py in base_python:
    for os_name in base_os:
        combo = {"os": os_name, "python-version": py}
        # 检查是否被 exclude
        excluded = False
        for ex in exclude:
            if all(k in combo and combo[k] == v for k, v in ex.items()):
                excluded = True
                break
        if not excluded:
            matrix.append(combo)

# 添加 include
matrix.extend(include)

for i, combo in enumerate(matrix, 1):
    print(f"  Job #{i}: Python {combo['python-version']} | {combo['os']}")
print(f"\n最终组合数: {len(matrix)}")
print("（如果不 exclude 3.10/ubuntu，会有 4 个组合）")

### 练习

写一个 Matrix 测试 Workflow：
- Python 3.10 / 3.11 / 3.12
- Ubuntu / macOS
- 每个组合中运行 `pytest`
- 加上 `fail-fast: false` 让各组合独立运行

---

## Day 9：依赖缓存

### 为什么需要缓存？

每次 CI 运行都会在一个**干净的 Runner** 上执行，这意味着每次都要重新 `pip install`。
如果你的依赖很多（比如 FastAPI + SQLAlchemy + asyncpg + ...），每次安装可能要花 2-3 分钟。
一天提交 10 次，就是 20-30 分钟的等待。

**缓存**可以让依赖只安装一次，后续运行直接从缓存恢复。

### actions/cache@v4

```yaml
- uses: actions/cache@v4
  with:
    path: ~/.cache/pip    # 要缓存的路径
    key: \${{ runner.os }}-pip-\${{ hashFiles('**/requirements.txt') }}
    restore-keys: |
      \${{ runner.os }}-pip-
```

**工作原理**：

```
第1次运行（cache miss）：
  key: ubuntu-pip-d41d8cd...（基于 requirements.txt 内容计算哈希）
  → 没找到缓存
  → 正常 pip install
  → 运行结束后自动保存缓存

第2次运行（requirements.txt 没变）：
  same key → cache HIT ✅
  → 跳过 pip install，直接恢复 ~/.cache/pip 目录
  → 耗时从 2min 降到 2s

第3次运行（改了 requirements.txt）：
  key 变了 → cache miss
  → 检查 restore-keys：ubuntu-pip- 有旧缓存
  → 恢复就缓存（部分命中），然后 pip install 只更新变更的包
```

In [ ]:
# 模拟缓存机制

import hashlib
import time

def simulate_cache(requirements_content, is_first_run=False):
    # 模拟 hashFiles('**/requirements.txt')
    key_content = hashlib.sha256(requirements_content.encode()).hexdigest()[:12]
    cache_key = f"ubuntu-pip-{key_content}"

    print(f"requirements.txt 内容哈希 → {key_content}")
    print(f"  Cache Key: {cache_key}")

    if is_first_run:
        print("  Status: ❌ CACHE MISS（首次运行，没有缓存）")
        print("  执行: pip install -r requirements.txt")
        time.sleep(0.3)
        print("  安装耗时: ~45 秒（模拟）")
        print("  完成: 缓存已保存")
        return cache_key
    else:
        print("  Status: ✅ CACHE HIT（缓存命中！）")
        print("  执行: 从缓存恢复依赖目录")
        time.sleep(0.1)
        print("  恢复耗时: ~2 秒（模拟）")
        return cache_key

print("=== 首次运行（无缓存） ===")
req1 = "flask==3.0.0\npytest==8.0.0\nruff==0.3.0"
key1 = simulate_cache(req1, is_first_run=True)

print("\n=== 第二次运行（requirements.txt 没变） ===")
key2 = simulate_cache(req1, is_first_run=False)
print(f"   Key 一致: {key1 == key2} → HIT!")

print("\n=== 第三次运行（改了 requirements.txt） ===")
req2 = "flask==3.1.0\npytest==8.1.0\nruff==0.4.0"
key3 = simulate_cache(req2, is_first_run=True)
print(f"   Key 不一致: {key2 != key3} → 新的 MISS")

### 不同语言的缓存路径

| 语言 | 缓存路径 |
|------|----------|
| Python (pip) | `~/.cache/pip` |
| npm | `~/.npm` |
| pipenv | `~/.local/share/virtualenvs` |
| Go | `~/go/pkg/mod` |
| Maven | `~/.m2/repository` |

### 练习

给之前的 Matrix 测试加上 pip 依赖缓存，对比首次运行和缓存命中后的耗时差异。

---

## Day 10：Artifacts（构建产物）

### 什么是 Artifacts？

Artifacts 是 Workflow 运行过程中产生的文件——测试报告、构建包、日志文件等。
你可以：
- **上传**（upload）：保存到 GitHub，供后续 Job 或其他地方使用
- **下载**（download）：在另一个 Job 中获取之前上传的产物
- **手动下载**：在 Actions 页面直接下载 zip

### 典型场景

```
Job A: 运行测试 + 生成报告
  ├── upload coverage report
  │
Job B: 构建 wheel 包
  ├── upload .whl 文件

→ 在 Actions 页面可以手动下载测试报告和 wheel 包
→ 或者 Job C download 报告，合并后生成全局报告
```

### 配置示例

```yaml
- name: Upload test results
  uses: actions/upload-artifact@v4
  with:
    name: test-results
    path: reports/
    retention-days: 30  # 保留 30 天（默认 90）

- name: Download test results
  uses: actions/download-artifact@v4
  with:
    name: test-results
    path: ./downloaded-reports
```

In [ ]:
# 模拟 Artifacts 的 upload/download

import os
import time

print("=" * 60)
print("模拟 Artifact 流程")
print("=" * 60)

# Step 1: 在 Job A 中生成报告
print("\n📦 Job A: 生成并上传 Artifact")
time.sleep(0.3)

# 模拟生成测试报告
report_content = """# Test Report
Date: 2026-05-21
Total: 42 tests
Passed: 40
Failed: 1
Skipped: 1
Duration: 3.2s
"""

# 模拟保存到文件
print("  生成报告文件: report.md")
print(f"  actions/upload-artifact@v4")
print(f"    name: unit-test-report")
print(f"    path: report.md")
print(f"    retention-days: 30")
print("  ✅ 上传成功！")

# Step 2: 在 Job B 中下载
print("\n📥 Job B: 下载并使用 Artifact")
time.sleep(0.2)
print("  actions/download-artifact@v4")
print(f"    name: unit-test-report")
print("  ✅ 下载成功，报告内容：")
print("-" * 30)
for line in report_content.strip().split('\n'):
    print(f"  {line}")
print("-" * 30)

### Artifact 保留策略

```yaml
# 设置保留天数
- uses: actions/upload-artifact@v4
  with:
    name: debug-logs
    retention-days: 7  # 调试日志保留 7 天就够了

- uses: actions/upload-artifact@v4
  with:
    name: release-build
    retention-days: 90  # 发布包保留 90 天
```

- 默认保留 **90 天**
- 可以设置 1 到 90 天
- 对于免费账户，总存储空间有限，建议及时清理不需要的 Artifact

---

## Day 11：Secrets 与 Variables 管理

### 仓库级 Secrets

Secrets 是敏感信息的仓库级配置——API 密钥、数据库密码、Token 等。

**设置路径**：
```
GitHub 仓库 → Settings → Secrets and variables → Actions
```
点击 "New repository secret"，输入名称和值。

**使用方式**：
```yaml
steps:
  - name: Deploy to production
    env:
      API_KEY: \${{ secrets.API_KEY }}
      DB_PASSWORD: \${{ secrets.DB_PASSWORD }}
    run: ./deploy.sh
```

**安全机制**：
- GitHub 在日志中**自动屏蔽** Secrets 的值（用 `***` 替代）
- 但是要注意：如果手动 `echo $MY_SECRET` 或拼接字符串，有可能绕过屏蔽
- 永远不要在日志中打印 Secret！

In [ ]:
# 模拟 Secrets 和 Variables 的使用

import os
import re

# 模拟 secrets 字典（在 GitHub 上这些从 Settings 页面设置）
secrets = {
    "API_KEY": "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
    "DB_PASSWORD": "my-super-secret-password",
    "DEPLOY_TOKEN": "ghp_xxxxxxxxxxxxxxxxxxxx"
}

# 模拟 variables
variables = {
    "DEPLOY_REGION": "us-east-1",
    "LOG_LEVEL": "info",
    "MAX_RETRIES": "3"
}

print("=" * 50)
print("Secrets & Variables 模拟")
print("=" * 50)

# 模拟在 workflow 中使用
print("\n🔐 Secrets（在日志中自动屏蔽）")
for name, value in secrets.items():
    # 模拟 GitHub 的日志屏蔽
    masked = re.sub(value, '***', f"Using {name}={value}")
    print(f"  {name}: {value}")  # 如果是在 Jupyter 打印
    print(f"  日志中的显示: Using {name}=***")

print("\n📋 Variables（非敏感，正常显示）")
print(f"  DEPLOY_REGION: {variables['DEPLOY_REGION']}")
print(f"  LOG_LEVEL: {variables['LOG_LEVEL']}")
print(f"  MAX_RETRIES: {variables['MAX_RETRIES']}")

print("\n⚠️ 安全提醒：永远不要在日志中直接打印 Secret 的值！")
print("   如果想验证 Secret 是否已设置，只打印长度：")
print(f"   API_KEY length: {len(secrets['API_KEY'])} characters")

### 环境级 Secrets

按 Environment 隔离不同环境的配置：

```yaml
jobs:
  deploy-dev:
    environment: dev
    runs-on: ubuntu-latest
    steps:
      - run: ./deploy.sh
        env:
          API_KEY: \${{ secrets.API_KEY }}  # dev 环境的 key

  deploy-prod:
    environment: prod
    runs-on: ubuntu-latest
    steps:
      - run: ./deploy.sh
        env:
          API_KEY: \${{ secrets.API_KEY }}  # prod 环境的 key（不同的值）
```

在 GitHub Settings → Environments 中配置每个环境的 Secrets。

### 练习

在仓库中设置一个 `TEST_TOKEN` Secret，写一个 Workflow 打印 Token 的长度（不要打印原始值）。

---

## Day 12：Pull Request 触发与 Status Check

### on: pull_request

除了 `on: push`，最常用的触发方式就是 `on: pull_request`：

```yaml
name: PR Check
on:
  pull_request:
    branches: [main]  # 只监控 main 分支的 PR
    types: [opened, synchronize, reopened]
    # opened: 创建 PR
    # synchronize: 向 PR 推送新 commit
    # reopened: 重新打开已关闭的 PR
```

### Status Check

当 PR 触发 Workflow 后，每个 Job 的状态会显示在 PR 页面：

- 🟢 绿色勾 → 通过
- 🔴 红色叉 → 失败
- 🟡 黄色圆 → 运行中

**Branch Protection 规则**：
在 Settings → Branches → Add branch protection rule 中设置 `main` 分支保护：

1. 勾选 "Require status checks to pass before merging"
2. 搜索并选择你的 Status Check（如 "test"）
3. 勾选 "Require branches to be up-to-date"

设置后：
- PR 页面会显示 "All checks must pass before merging"
- 不满足条件的 PR **不能 merge**（按钮灰色）
- 这就保证了 **main 分支的代码永远是通过测试的**

In [ ]:
# 模拟 PR Status Check

print("=" * 60)
print("Pull Request Status Check 模拟")
print("=" * 60)

pr_checks = {
    "lint": "✅ success",
    "test (3.10, ubuntu)": "✅ success",
    "test (3.11, ubuntu)": "✅ success",
    "test (3.12, ubuntu)": "✅ success",
    "test (3.12, macos)": "🔄 running",
    "build": "⏳ waiting",
}

print("\n📋 PR #42: fix/login-bug")
print("   base: main ← head: feature/fix-login")
print("   Latest commit: a1b2c3d")
print("\n🔍 Status Checks:")
print("-" * 40)

all_passing = True
for check, status in pr_checks.items():
    print(f"  {status}  {check}")
    if status.startswith("❌"):
        all_passing = False

print("-" * 40)
if all_passing:
    print("✅ All checks passed — Merge button is GREEN 🟢")
    print("   This PR is ready to merge!")
else:
    print("❌ Some checks are failing — Merge button is GRAY")
    print("   Fix the failing checks before merging!")

print("\n⚠️ With Branch Protection enabled:")
print("   - Cannot merge if checks are failing")
print("   - Cannot push directly to main")
print("   - PR must be reviewed before merge (optional)")

### 练习

1. 写一个 PR 触发的 Workflow（lint + test）
2. 在 Settings → Branches 设置 main 分支保护，要求 CI 全部通过才能 merge
3. 创建一个 PR，观察 Status Check 的执行

---

## Day 13：定时任务与手动触发

### 定时触发（schedule）

```yaml
on:
  schedule:
    - cron: "0 2 * * *"    # 每天 UTC 2:00（北京时间 10:00）
    - cron: "30 6 * * 1"   # 每周一 UTC 6:30
```

**Cron 语法**：

```
┌───────────── 分钟 (0-59)
│ ┌───────────── 小时 (0-23)
│ │ ┌───────────── 日 (1-31)
│ │ │ ┌───────────── 月 (1-12)
│ │ │ │ ┌───────────── 星期 (0-6, 0=周日)
│ │ │ │ │
* * * * *
```

**常用定时**：
| 时间 | Cron | 说明 |
|------|------|------|
| 每天凌晨 2 点 | `0 2 * * *` | 每天跑一次完整测试 |
| 每周末 | `0 6 * * 0` | 每周日早 6 点 |
| 每小时 | `0 * * * *` | 每小时跑一次 |
| 工作日每 4 小时 | `0 */4 * * 1-5` | 周一到周五每 4 小时 |

### 手动触发（workflow_dispatch）

```yaml
on:
  workflow_dispatch:
    inputs:
      environment:
        description: '部署环境'
        required: true
        default: 'staging'
        type: choice
        options:
          - dev
          - staging
          - prod
      version:
        description: '版本号'
        required: true
        type: string
```

使用 `\${{ inputs.environment }}` 和 `\${{ inputs.version }}` 获取输入值。

In [ ]:
# 模拟 schedule 和 workflow_dispatch

from datetime import datetime, timezone
import time

print("=" * 50)
print("定时任务 & 手动触发")
print("=" * 50)

# 模拟 schedule
print("\n⏰ Schedule 触发")
now = datetime.now(timezone.utc)
print(f"  当前 UTC 时间: {now.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  配置: cron: '0 2 * * *'")
print(f"  下次触发: 明天 UTC 02:00")
print(f"  运作: nightly-test (运行全部测试套件)")

# 模拟 workflow_dispatch
print("\n👆 Workflow Dispatch 触发")
print("  手动触发！在 Actions 页面点击 'Run workflow' 按钮")
print("\n  输入参数:")
print("    environment: staging")
print("    version: v2.1.0")
print("    run-smoke-tests: true")
print("\n  ✅ 使用参数:")
print("    Deploying version v2.1.0 to staging...")
time.sleep(0.5)
print("    Running smoke tests...")
time.sleep(0.3)
print("    ✅ Deployment to staging completed!")

### 练习

1. 写一个定时 Workflow，每天凌晨 2 点运行一次完整测试
2. 加上手动触发选项，可指定运行哪些测试（单元测试 / 集成测试 / 全部）

---

## 🎯 第2周总结

### 本周学到的核心技能

| 技能 | 关键命令/配置 |
|------|---------------|
| Matrix 策略 | `strategy.matrix.python-version: ["3.10", "3.11", "3.12"]` |
| 缓存加速 | `actions/cache@v4` + `hashFiles` |
| Artifact | `actions/upload-artifact@v4` / `actions/download-artifact@v4` |
| Secrets | `\${{ secrets.NAME }}`，在 Settings 中设置 |
| PR 触发 | `on: pull_request` + Branch Protection |
| 定时任务 | `on: schedule: - cron: '0 2 * * *'` |
| 手动触发 | `on: workflow_dispatch` + `inputs` |

### 最佳实践

1. **Matrix 控制数量**：别让组合爆炸，用 `exclude` 排除不必要的组合
2. **缓存要打对 key**：`hashFiles` 对依赖文件变更敏感，不变时复用缓存
3. **Artifact 设置保留期**：临时文件 7 天，发布包 90 天
4. **Secrets 永不打印**：在日志中验证 Secrets 时只检查长度或是否为空
5. **Branch Protection**：main 分支必须保护，确保 CI 通过才能 merge

---

## 🧪 综合练习：完整的 Python 包 CI 流水线

构建一个完整的 Python 包 CI 流水线，涵盖本周所有知识点：

```
push / PR 触发
  ├── Job: matrix-test
  │     ├── strategy: Python 3.10/3.11/3.12 × ubuntu/macos
  │     ├── pip 缓存
  │     ├── 运行 pytest
  │     └── upload coverage artifact
  ├── Job: lint
  │     └── ruff check + mypy
  ├── Job: build
  │     ├── 构建 Python wheel 包
  │     └── upload wheel artifact
  └── Job: coverage-report (依赖 test)
        ├── download coverage artifacts
        └── 生成 HTML coverage 报告 + upload
```

### 配置概览

```yaml
name: Python Package CI
on: [push, pull_request]

jobs:
  matrix-test:
    strategy:
      fail-fast: false
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
        os: [ubuntu-latest, macos-latest]
    runs-on: \${{ matrix.os }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: \${{ matrix.python-version }}
      - uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: \${{ runner.os }}-pip-\${{ hashFiles('**/requirements.txt') }}
      - run: pip install -r requirements.txt
      - run: pytest --cov-report=xml
      - uses: actions/upload-artifact@v4
        with:
          name: coverage-\${{ matrix.os }}-\${{ matrix.python-version }}
          path: coverage.xml

  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install ruff mypy
      - run: ruff check .
      - run: mypy .

  build:
    needs: [matrix-test, lint]
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install build
      - run: python -m build
      - uses: actions/upload-artifact@v4
        with:
          name: dist
          path: dist/

  coverage-report:
    needs: [matrix-test]
    if: always()
    runs-on: ubuntu-latest
    steps:
      - uses: actions/download-artifact@v4
      - uses: ...
      - run: ...  # 合并 coverage 报告
```

In [ ]:
# 模拟完整 CI 流水线

import time

print("=" * 60)
print("🚀 Python Package CI Pipeline")
print("=" * 60)

pipeline = [
    ("matrix-test (3.10, ubuntu)", None, 1.0, "upload: coverage-ubuntu-3.10"),
    ("matrix-test (3.11, ubuntu)", None, 1.0, "upload: coverage-ubuntu-3.11"),
    ("matrix-test (3.12, ubuntu)", None, 1.0, "upload: coverage-ubuntu-3.12"),
    ("matrix-test (3.10, macos)", None, 1.2, "upload: coverage-macos-3.10"),
    ("matrix-test (3.11, macos)", None, 1.2, "upload: coverage-macos-3.11"),
    ("matrix-test (3.12, macos)", None, 1.2, "upload: coverage-macos-3.12"),
    ("lint", None, 0.6, None),
]

completed_jobs = set()

for name, dep, duration, artifact in pipeline:
    if dep:
        while dep not in completed_jobs:
            pass
    print(f"\n▶ Running: {name}")
    time.sleep(duration)
    print(f"  ✅ 完成")
    if artifact:
        print(f"  📦 {artifact}")
    completed_jobs.add(name)

# build and coverage depend on matrix-test and lint
print("\n▶ Running: build (依赖 matrix-test + lint)")
time.sleep(0.8)
print("  ✅ python -m build 成功")
print("  📦 upload: dist/ (wheel + source dist)")
completed_jobs.add("build")

print("\n▶ Running: coverage-report (依赖 matrix-test)")
time.sleep(0.5)
print("  ✅ 合并所有 coverage 报告")
print("  ✅ 生成 HTML 报告")
print("  📦 upload: coverage-html/")
completed_jobs.add("coverage-report")

print("\n" + "=" * 60)
print("🎉 完整 CI Pipeline 执行完成！")
print(f"   共运行 {len(pipeline) + 2} 个 Job")
print("=" * 60)